# Assignment 2: Stock Market Analysis Agent

This notebook tests the stock analysis LangGraph agent (`stock_agent/`).

Workflow nodes:
1. `fetch_stock_data`: Fetches 60 days of daily history using `yfinance`.
2. `calculate_indicators`: Calculates 10-day SMA, 20-day SMA, and 14-day RSI.
3. `generate_recommendation`: Produces a `BUY`, `HOLD`, or `SELL` recommendation and text report.

## 1. Test Indicator Calculations

In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath("stock_agent"))

from components.helper_functions import (
    calculate_sma,
    calculate_rsi,
    evaluate_recommendation,
)

steps = np.array([1.2, -0.7, 1.4, -0.8, 1.1, 0.9, -1.0, 1.3, -0.6, 1.1] * 6)
sample_prices = pd.Series(180.0 + np.cumsum(steps))

sma_10 = calculate_sma(sample_prices, 10)
sma_20 = calculate_sma(sample_prices, 20)
rsi_14 = calculate_rsi(sample_prices, 14)
rec = evaluate_recommendation(float(sample_prices.iloc[-1]), sma_10, sma_20, rsi_14)

print(f"Latest Close: ${sample_prices.iloc[-1]:.2f}")
print(f"10-Day SMA: ${sma_10:.2f}")
print(f"20-Day SMA: ${sma_20:.2f}")
print(f"14-Day RSI: {rsi_14:.2f}")
print(f"Recommendation: {rec['action']} ({rec['confidence']})")

Latest Close: $203.40
10-Day SMA: $201.72
20-Day SMA: $199.77
14-Day RSI: 66.67
Recommendation: BUY (Moderate)

## 2. Run Agent Across BUY, HOLD, and SELL Scenarios

In [2]:
from unittest.mock import patch, MagicMock
import components.nodes as stock_nodes
from graph import stock_agent
from components.state import StockAgentState

dates_60d = pd.bdate_range(end="2026-10-23", periods=60)

def build_df(close_arr):
    close_arr = np.array(close_arr, dtype=float)
    return pd.DataFrame({
        "Open": close_arr * 0.995,
        "High": close_arr * 1.012,
        "Low": close_arr * 0.988,
        "Close": close_arr,
        "Volume": np.linspace(42_000_000, 58_000_000, len(close_arr)).astype(int),
    }, index=dates_60d)

aapl_closes = 205.0 + np.cumsum(np.array([1.2, -0.7, 1.4, -0.8, 1.1, 0.9, -1.0, 1.3, -0.6, 1.1] * 6))
msft_closes = 415.0 + np.cumsum(np.concatenate([np.array([0.5, -0.4, 0.6, -0.3, 0.4] * 9), np.array([2.2, 1.8, -0.3, 2.5, 2.0] * 3)]))
tsla_closes = 265.0 + np.cumsum(np.array([-1.3, 0.8, -1.5, 0.7, -1.1, -0.9, 1.0, -1.4, 0.6, -1.2] * 6))

sample_data = {
    "AAPL": build_df(aapl_closes),
    "MSFT": build_df(msft_closes),
    "TSLA": build_df(tsla_closes),
}

class DummyTicker:
    def __init__(self, symbol):
        self.symbol = symbol.upper()

    def history(self, period="60d", interval="1d"):
        return sample_data.get(self.symbol, pd.DataFrame())

mock_yf = MagicMock()
mock_yf.Ticker = DummyTicker

with patch.object(stock_nodes, "yf", mock_yf):
    for symbol in ["AAPL", "MSFT", "TSLA"]:
        state = StockAgentState(
            ticker=symbol,
            stock_data=None,
            indicators=None,
            recommendation=None,
            analysis_report=None,
            error=None,
        )
        res = stock_agent.invoke(state)
        print(res["analysis_report"])
        print()

Stock Analysis Report: AAPL
Period: 2026-08-03 to 2026-10-23 (60 trading days)

Price Summary:
- Current Price: $228.40 (+0.48% today)
- 60-Day Change: +10.77%
- 60-Day Range: $203.03 to $231.14
- Avg Daily Volume: 49,999,999

Technical Indicators:
- 10-Day SMA: $226.72
- 20-Day SMA: $224.77
- 14-Day RSI: 66.67
- Trend Signal: Bullish (SMA-10 > SMA-20)
- Momentum Signal: Neutral (RSI = 66.67)

Recommendation: BUY (Confidence: Moderate)

Notes:
- 10-day SMA ($226.72) is above 20-day SMA ($224.77) by +0.87%.
- Current price ($228.40) is above both moving averages.
- 14-day RSI (66.67) is in the normal 30-70 range.

Stock Analysis Report: MSFT
Period: 2026-08-03 to 2026-10-23 (60 trading days)

Price Summary:
- Current Price: $446.80 (+0.45% today)
- 60-Day Change: +7.53%
- 60-Day Range: $410.12 to $452.16
- Avg Daily Volume: 49,999,999

Technical Indicators:
- 10-Day SMA: $439.36
- 20-Day SMA: $431.92
- 14-Day RSI: 96.28
- Trend Signal: Bullish (SMA-10 > SMA-20)
- Momentum Signal: Overbo

## 3. Error Handling Check (Invalid Ticker & Empty Input)

In [3]:
with patch.object(stock_nodes, "yf", mock_yf):
    for bad_input in ["INVALID_TICKER", "   "]:
        res = stock_agent.invoke({
            "ticker": bad_input,
            "stock_data": None,
            "indicators": None,
            "recommendation": None,
            "analysis_report": None,
            "error": None,
        })
        print(res["analysis_report"])
        print()

Stock Analysis Report: INVALID_TICKER
Status: Error
Details: No historical stock data found for ticker 'INVALID_TICKER'.

Stock Analysis Report:    
Status: Error
Details: Invalid ticker symbol: Ticker cannot be empty.
